# 03_Business_Intelligence_Dashboard
## Analisis Pendapatan Kepala Keluarga RT 013 / RW 004 Dusun Ciranggon

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

COLORS = {
    'primary':   '#2E86AB',
    'secondary': '#A23B72',
    'accent':    '#F18F01',
    'red':       '#C73E1D',
    'green':     '#44994A',
    'gray':      '#6C757D',
    'light':     '#F8F9FA',
}

df = pd.read_csv('../data/pendapatan_rt013_clean.csv')
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')

---
## Dashboard BI

In [ ]:
fig = plt.figure(figsize=(18, 24))
gs = gridspec.GridSpec(8, 6, figure=fig, hspace=0.4, wspace=0.35,
                       height_ratios=[0.6, 0.6, 1, 1, 1, 1, 1, 0.5])

# ── Judul Dashboard ──
ax_title = fig.add_subplot(gs[0, :])
ax_title.axis('off')
ax_title.text(0.5, 0.65, 'BUSINESS INTELLIGENCE DASHBOARD',
              fontsize=20, fontweight='bold', ha='center', va='center',
              color=COLORS['primary'])
ax_title.text(0.5, 0.25,
              'Analisis Pendapatan Kepala Keluarga RT 013 / RW 004 Dusun Ciranggon',
              fontsize=12, ha='center', va='center', color=COLORS['gray'])

# ── KPI Cards ──
kpis = [
    ('Jumlah KK',       f"{len(df)}",                        COLORS['primary']),
    ('Total Pendapatan', f"Rp{df['pendapatan'].sum():,.0f}", COLORS['secondary']),
    ('Rata-rata',        f"Rp{df['pendapatan'].mean():,.0f}", COLORS['accent']),
    ('Maksimum',         f"Rp{df['pendapatan'].max():,}",     COLORS['green']),
    ('Minimum',          f"Rp{df['pendapatan'].min():,}",     COLORS['red']),
]

for i, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor(color)
    ax.text(0.5, 0.7, value, fontsize=18, fontweight='bold',
            ha='center', va='center', color='white')
    ax.text(0.5, 0.2, label, fontsize=10,
            ha='center', va='center', color='white', alpha=0.9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

# ── 1. Distribusi Pekerjaan ──
ax1 = fig.add_subplot(gs[2, :3])
pekerjaan_counts = df['pekerjaan'].value_counts()
colors_bar = [COLORS['primary'], COLORS['green'], COLORS['accent']]
bars1 = ax1.bar(pekerjaan_counts.index, pekerjaan_counts.values,
                color=colors_bar, edgecolor='white', linewidth=0.8)
ax1.bar_label(bars1, padding=2, fontsize=10, fontweight='bold')
ax1.set_title('Distribusi Pekerjaan', fontsize=13, fontweight='bold', pad=10)
ax1.set_ylabel('Jumlah KK')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── 2. Distribusi Pendidikan ──
ax2 = fig.add_subplot(gs[2, 3:])
pend_order = ['Tidak Sekolah', 'SD/Sederajat', 'SMP/Sederajat', 'SMA/Sederajat']
pend_counts = df['pendidikan'].value_counts().reindex(pend_order).dropna()
colors_pend = [COLORS['red'], COLORS['green'], COLORS['primary'], COLORS['secondary']]
bars2 = ax2.bar(pend_counts.index, pend_counts.values,
                color=colors_pend, edgecolor='white', linewidth=0.8)
ax2.bar_label(bars2, padding=2, fontsize=10, fontweight='bold')
ax2.set_title('Distribusi Pendidikan', fontsize=13, fontweight='bold', pad=10)
ax2.set_ylabel('Jumlah KK')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── 3. Histogram Pendapatan ──
ax3 = fig.add_subplot(gs[3, :3])
ax3.hist(df['pendapatan'] / 1_000_000, bins=10, edgecolor='white',
         color=COLORS['primary'], linewidth=0.8)
ax3.axvline(df['pendapatan'].mean() / 1_000_000, color=COLORS['red'],
            linestyle='--', linewidth=1.5, label=f"Mean: {df['pendapatan'].mean()/1e6:.1f}jt")
ax3.axvline(df['pendapatan'].median() / 1_000_000, color=COLORS['green'],
            linestyle='--', linewidth=1.5, label=f"Median: {df['pendapatan'].median()/1e6:.1f}jt")
ax3.set_title('Histogram Pendapatan', fontsize=13, fontweight='bold', pad=10)
ax3.set_xlabel('Pendapatan (juta Rupiah)')
ax3.set_ylabel('Jumlah KK')
ax3.legend(fontsize=9)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# ── 4. Pendapatan per Pekerjaan ──
ax4 = fig.add_subplot(gs[3, 3:])
pekerjaan_order = df.groupby('pekerjaan')['pendapatan'].median().sort_values().index
box_data = [df[df['pekerjaan'] == p]['pendapatan'] / 1_000_000 for p in pekerjaan_order]
bp = ax4.boxplot(box_data, tick_labels=pekerjaan_order, patch_artist=True)
for patch, color in zip(bp['boxes'], [COLORS['accent'], COLORS['green'], COLORS['primary']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax4.set_title('Pendapatan per Pekerjaan', fontsize=13, fontweight='bold', pad=10)
ax4.set_ylabel('Pendapatan (juta Rupiah)')
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

# ── 5. Status Rumah ──
ax5 = fig.add_subplot(gs[4, :3])
status_counts = df['status_rumah'].value_counts()
colors_status = [COLORS['primary'], COLORS['accent'], COLORS['red']]
bars5 = ax5.bar(status_counts.index, status_counts.values,
                color=colors_status, edgecolor='white', linewidth=0.8)
ax5.bar_label(bars5, padding=2, fontsize=10, fontweight='bold')
ax5.set_title('Status Rumah', fontsize=13, fontweight='bold', pad=10)
ax5.set_ylabel('Jumlah KK')
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

# ── 6. Bantuan Sosial ──
ax6 = fig.add_subplot(gs[4, 3:])
bantuan_order = ['Tidak Ada', 'PKH', 'BPNT']
bantuan_counts = df['bantuan_sosial'].value_counts().reindex(bantuan_order).dropna()
colors_bantuan = [COLORS['gray'], COLORS['green'], COLORS['accent']]
bars6 = ax6.bar(bantuan_counts.index, bantuan_counts.values,
                color=colors_bantuan, edgecolor='white', linewidth=0.8)
ax6.bar_label(bars6, padding=2, fontsize=10, fontweight='bold')
ax6.set_title('Bantuan Sosial', fontsize=13, fontweight='bold', pad=10)
ax6.set_ylabel('Jumlah KK')
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)

# ── Ringkasan Dashboard ──
ax_summary = fig.add_subplot(gs[5:7, :])
ax_summary.axis('off')

total_kk = len(df)
mean_pend = df['pendapatan'].mean()
median_pend = df['pendapatan'].median()
total_pend = df['pendapatan'].sum()
max_pend = df['pendapatan'].max()
min_pend = df['pendapatan'].min()
dominant_job = df['pekerjaan'].mode()[0]
dominant_edu = df['pendidikan'].mode()[0]
mean_tanggungan = df['jumlah_tanggungan'].mean()
mean_umur = df['umur'].mean()
mean_lama = df['lama_bekerja'].mean()

under_umr = (df['pendapatan'] < 4_800_000).sum()
pct_under_umr = under_umr / total_kk * 100
milik_sendiri = (df['status_rumah'] == 'Milik Sendiri').sum()
terima_bantuan = (df['bantuan_sosial'] != 'Tidak Ada').sum()

summary_lines = [
    'INSIGHT DASHBOARD',
    '=' * 60,
    f'1. Dataset mencakup {total_kk} Kepala Keluarga di RT 013 dengan total pendapatan Rp{total_pend:,.0f}.' if total_kk else '',
    f'2. Rata-rata pendapatan KK adalah Rp{mean_pend:,.0f} dengan median Rp{median_pend:,.0f}, '
    f'selisih Rp{mean_pend - median_pend:,.0f} mengindikasikan distribusi yang menceng ke kanan (beberapa KK berpenghasilan tinggi).' if True else '',
    f'3. Pekerjaan dominan adalah {dominant_job} ({pekerjaan_counts[dominant_job]} KK), '
    f'diikuti Petani ({pekerjaan_counts.get("Petani", 0)} KK) dan Pedagang ({pekerjaan_counts.get("Pedagang", 0)} KK).' if True else '',
    f'4. Pendidikan terbanyak adalah {dominant_edu} ({pend_counts.get(dominant_edu, 0)} KK), '
    f'namun masih terdapat KK yang tidak bersekolah ({pend_counts.get("Tidak Sekolah", 0)} KK).' if True else '',
    f'5. Sebanyak {under_umr} KK ({pct_under_umr:.0f}%) memiliki pendapatan di bawah UMR Kab. Karawang (Rp4.800.000).' if True else '',
    f'6. Sebanyak {milik_sendiri} KK ({milik_sendiri/total_kk*100:.0f}%) memiliki rumah sendiri, '
    f'sisanya {total_kk - milik_sendiri} KK masih kontrak atau numpang.' if True else '',
    f'7. Penerima bantuan sosial tercatat {terima_bantuan} KK, '
    f'terdiri dari PKH {(df["bantuan_sosial"]=="PKH").sum()} KK dan BPNT {(df["bantuan_sosial"]=="BPNT").sum()} KK.' if True else '',
    f'8. Rata-rata jumlah tanggungan {mean_tanggungan:.1f} orang per KK, '
    f'dengan rata-rata usia KK {mean_umur:.0f} tahun dan lama bekerja {mean_lama:.0f} tahun.' if True else '',
]

summary_text = '\n'.join(line for line in summary_lines if line)
ax_summary.text(0.02, 0.95, summary_text, fontsize=11,
                ha='left', va='top', fontfamily='monospace',
                linespacing=1.6)

# ── Footer ──
ax_footer = fig.add_subplot(gs[7, :])
ax_footer.axis('off')
ax_footer.text(0.5, 0.5,
               'Data Warehouse & Business Intelligence  |  UAS  |  Data Dummy RT 013 / RW 004 Dusun Ciranggon',
               fontsize=9, ha='center', va='center', color=COLORS['gray'], style='italic')

plt.savefig('../charts/dashboard_bi.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Dashboard berhasil disimpan ke charts/dashboard_bi.png')